In [125]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout


In [126]:
df=pd.read_csv(r'C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\final_processed_data.csv')
df1=pd.read_csv(r'C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\batsman_profiles.csv')

In [127]:
df['Batsman_Name_clean'] = df['Full Name'].astype(str).str.lower().str.strip()
df1['Full_Name_clean'] = df1['Full Name'].astype(str).str.lower().str.strip()

In [128]:
for col in ['isFour', 'isSix', 'isWicket']:
    df[col] = df[col].astype(int)


In [129]:
df['run'] = pd.to_numeric(df['run'], errors='coerce').fillna(0)

In [130]:
df['pitchLine_code'] = df['pitchLine'].astype('category').cat.codes
df['pitchLength_code'] = df['pitchLength'].astype('category').cat.codes

In [131]:
pitchLine_mapping = dict(enumerate(df['pitchLine'].astype('category').cat.categories))
pitchLength_mapping = dict(enumerate(df['pitchLength'].astype('category').cat.categories))


In [132]:
df['isBoundary'] = df['isFour'] | df['isSix'] 

In [133]:
df.columns

Index(['Unnamed: 0', 'match_obj_id', 'inningNumber', 'oversActual',
       'pitchLine', 'pitchLength', 'isFour', 'isSix', 'isWicket', 'byes',
       'legbyes', 'wides', 'noballs', 'run', 'totalRuns', 'totalWickets',
       'shotType', 'time_of_day', 'Ground Name', 'Batsman_Name',
       'Batsman_Role', 'Full Name', 'Batsman_Batting_Style',
       'Batsman_Playing_Role', 'Bowler_Name', 'Bowler_Role',
       'Full Name_bowler', 'Batting Style (s)_bowler', 'Bowler_Bowling_Style',
       'Bowler_Playing_Role', 'Batsman_Name_clean', 'pitchLine_code',
       'pitchLength_code', 'isBoundary'],
      dtype='object')

In [134]:
agg_cols = ['run', 'isFour', 'isSix', 'isWicket', 'isBoundary', 'pitchLine_code', 'pitchLength_code']
batsman_stats = df.groupby('Batsman_Name_clean')[agg_cols].agg({
    'run': ['sum', 'mean'],
    'isFour': 'sum',
    'isSix': 'sum',
    'isWicket': 'sum',
    'isBoundary': 'sum',
    'pitchLine_code': lambda x: x.mode()[0] if not x.mode().empty else np.nan,
    'pitchLength_code': lambda x: x.mode()[0] if not x.mode().empty else np.nan
}).reset_index()

In [135]:
batsman_stats.columns = [
    'Batsman_Name_clean',
    'total_runs', 'avg_runs', 'total_4s', 'total_6s',
    'total_wickets', 'total_boundaries', 'mode_pitchLine', 'mode_pitchLength'
]

In [136]:
balls_faced = df.groupby('Batsman_Name_clean').size().rename('balls_faced')
batsman_stats = batsman_stats.merge(balls_faced, on='Batsman_Name_clean', how='left')

In [137]:
batsman_stats['strike_rate'] = batsman_stats['total_runs'] / batsman_stats['balls_faced'] * 100
batsman_stats['boundary_rate'] = batsman_stats['total_boundaries'] / batsman_stats['balls_faced']


In [138]:
batsman_stats['Batsman_Name_clean']

0         aamir kaleem
1          aaron finch
2        aaron johnson
3          aaron jones
4       aaron phangiso
            ...       
457       yuvraj singh
458        zahoor khan
459         zane green
460        zawar farid
461    zeeshan maqsood
Name: Batsman_Name_clean, Length: 462, dtype: object

In [139]:
batsman_stats['dot_ball_pct'] = df.groupby('Batsman_Name_clean')['run'].apply(lambda x: (x==0).sum() / x.count() * 100)


In [140]:
batsman_stats['dismissal_ratio'] = batsman_stats['total_wickets'] / batsman_stats['balls_faced']

In [141]:
df_features = pd.merge(
    df1,
    batsman_stats,
    left_on='Full_Name_clean',
    right_on='Batsman_Name_clean',
    how='left'
)

In [142]:
def get_phase(over):
    if over <= 6:
        return 'Powerplay'
    elif over <= 15:
        return 'Middle'
    else:
        return 'Death'

df['phase'] = df['oversActual'].apply(get_phase)

phase_perf = df.groupby(['Batsman_Name_clean', 'phase'])['run'].mean().unstack().fillna(0)
phase_perf.columns = [f'avg_runs_{c.lower()}' for c in phase_perf.columns]
batsman_stats = batsman_stats.merge(phase_perf, left_index=True, right_index=True, how='left')


In [143]:
batsman_stats['dismissal_ratio'] = batsman_stats['total_wickets'] / batsman_stats['balls_faced']


In [144]:
consistency = df.groupby('Batsman_Name_clean')['run'].std().rename('run_std')
batsman_stats = batsman_stats.merge(consistency, left_index=True, right_index=True, how='left')
batsman_stats['consistency_index'] = batsman_stats['avg_runs'] / (batsman_stats['run_std'] + 1e-6)


In [145]:
bowler_effect = df.groupby(['Batsman_Name_clean', 'Bowler_Bowling_Style'])['run'].mean().unstack().fillna(0)
bowler_effect.columns = [f'avg_vs_{c.replace(" ", "_").lower()}' for c in bowler_effect.columns]
batsman_stats = batsman_stats.merge(bowler_effect, left_index=True, right_index=True, how='left')


In [146]:
tod_perf = df.groupby(['Batsman_Name_clean', 'time_of_day'])['run'].mean().unstack().fillna(0)
tod_perf.columns = [f'avg_runs_{c.lower()}' for c in tod_perf.columns]
batsman_stats = batsman_stats.merge(tod_perf, left_index=True, right_index=True, how='left')


In [147]:
batsman_stats.columns

Index(['Batsman_Name_clean', 'total_runs', 'avg_runs', 'total_4s', 'total_6s',
       'total_wickets', 'total_boundaries', 'mode_pitchLine',
       'mode_pitchLength', 'balls_faced', 'strike_rate', 'boundary_rate',
       'dot_ball_pct', 'dismissal_ratio', 'avg_runs_death', 'avg_runs_middle',
       'avg_runs_powerplay', 'run_std', 'consistency_index',
       'avg_vs_left-arm_fast', 'avg_vs_left-arm_fast-medium',
       'avg_vs_left-arm_medium',
       'avg_vs_left-arm_medium,slow_left-arm_orthodox',
       'avg_vs_left-arm_medium-fast', 'avg_vs_left-arm_wrist-spin',
       'avg_vs_legbreak', 'avg_vs_legbreak_googly', 'avg_vs_right-arm_fast',
       'avg_vs_right-arm_fast-medium', 'avg_vs_right-arm_fast-medium,legbreak',
       'avg_vs_right-arm_fast-medium,legbreak_googly',
       'avg_vs_right-arm_medium', 'avg_vs_right-arm_medium,legbreak',
       'avg_vs_right-arm_medium,right-arm_offbreak',
       'avg_vs_right-arm_medium-fast',
       'avg_vs_right-arm_medium-fast,right-arm_offbr

In [148]:
batsman_stats.to_csv(r'C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\batsman_stats1.csv')

In [149]:
df.columns

Index(['Unnamed: 0', 'match_obj_id', 'inningNumber', 'oversActual',
       'pitchLine', 'pitchLength', 'isFour', 'isSix', 'isWicket', 'byes',
       'legbyes', 'wides', 'noballs', 'run', 'totalRuns', 'totalWickets',
       'shotType', 'time_of_day', 'Ground Name', 'Batsman_Name',
       'Batsman_Role', 'Full Name', 'Batsman_Batting_Style',
       'Batsman_Playing_Role', 'Bowler_Name', 'Bowler_Role',
       'Full Name_bowler', 'Batting Style (s)_bowler', 'Bowler_Bowling_Style',
       'Bowler_Playing_Role', 'Batsman_Name_clean', 'pitchLine_code',
       'pitchLength_code', 'isBoundary', 'phase'],
      dtype='object')

In [150]:
df['ball_type'] = df['pitchLength'].astype(str) + "_" + df['pitchLine'].astype(str)


In [151]:
ball_mean = df.groupby(['Batsman_Name_clean', 'ball_type'])['run'].mean().reset_index()


In [152]:
# Find the ball type with minimum mean runs (best to bowl)
best_ball = ball_mean.loc[ball_mean.groupby('Batsman_Name_clean')['run'].idxmin()]

# Rename columns for clarity
best_ball = best_ball.rename(columns={'ball_type': 'best_ball_type', 'run': 'min_mean_runs'})

# Merge with batsman stats
df_features = df_features.merge(best_ball[['Batsman_Name_clean', 'best_ball_type']], 
                                left_on='Full_Name_clean', right_on='Batsman_Name_clean', how='left')


In [153]:
df_features.head()

,Full Name,Run_Scored,Balls_Faced,Strike_Rate,Dismissals,Dismissal_Rate,Boundary_percentage,Average,Full_Name_clean,Batsman_Name_clean_x,...,total_boundaries,mode_pitchLine,mode_pitchLength,balls_faced,strike_rate,boundary_rate,dot_ball_pct,dismissal_ratio,Batsman_Name_clean_y,best_ball_type
0,AB de Villiers,117.0,61.0,191.803279,3.0,20.333333,22.950820,39.000000,ab de villiers,ab de villiers,...,14,2,2,65,180.000000,0.215385,NaN,0.046154,ab de villiers,FULL_DOWN_LEG
1,Aamir Kaleem,0.0,5.0,0.000000,2.0,2.500000,0.000000,0.000000,aamir kaleem,aamir kaleem,...,0,2,2,5,0.000000,0.000000,NaN,0.400000,aamir kaleem,FULL_OUTSIDE_OFFSTUMP
2,Aaron Finch,333.0,258.0,129.069767,11.0,23.454545,13.953488,30.272727,aaron finch,aaron finch,...,36,2,2,278,119.784173,0.129496,NaN,0.039568,aaron finch,FULL_TOSS_WIDE_OUTSIDE_OFFSTUMP
3,Aaron Johnson,101.0,73.0,138.356164,4.0,18.250000,21.917808,25.250000,aaron johnson,aaron johnson,...,16,2,2,75,134.666667,0.213333,NaN,0.053333,aaron johnson,FULL_TOSS_OUTSIDE_OFFSTUMP
4,Aaron Jones,152.0,87.0,174.712644,1.0,87.000000,21.839080,152.000000,aaron jones,aaron jones,...,19,2,2,93,163.440860,0.204301,NaN,0.010753,aaron jones,YORKER_OUTSIDE_OFFSTUMP


In [154]:
df_features.columns

Index(['Full Name', 'Run_Scored', 'Balls_Faced', 'Strike_Rate', 'Dismissals',
       'Dismissal_Rate', 'Boundary_percentage', 'Average', 'Full_Name_clean',
       'Batsman_Name_clean_x', 'total_runs', 'avg_runs', 'total_4s',
       'total_6s', 'total_wickets', 'total_boundaries', 'mode_pitchLine',
       'mode_pitchLength', 'balls_faced', 'strike_rate', 'boundary_rate',
       'dot_ball_pct', 'dismissal_ratio', 'Batsman_Name_clean_y',
       'best_ball_type'],
      dtype='object')

In [155]:
df.columns

Index(['Unnamed: 0', 'match_obj_id', 'inningNumber', 'oversActual',
       'pitchLine', 'pitchLength', 'isFour', 'isSix', 'isWicket', 'byes',
       'legbyes', 'wides', 'noballs', 'run', 'totalRuns', 'totalWickets',
       'shotType', 'time_of_day', 'Ground Name', 'Batsman_Name',
       'Batsman_Role', 'Full Name', 'Batsman_Batting_Style',
       'Batsman_Playing_Role', 'Bowler_Name', 'Bowler_Role',
       'Full Name_bowler', 'Batting Style (s)_bowler', 'Bowler_Bowling_Style',
       'Bowler_Playing_Role', 'Batsman_Name_clean', 'pitchLine_code',
       'pitchLength_code', 'isBoundary', 'phase', 'ball_type'],
      dtype='object')

In [156]:
batsman_stats.columns

Index(['Batsman_Name_clean', 'total_runs', 'avg_runs', 'total_4s', 'total_6s',
       'total_wickets', 'total_boundaries', 'mode_pitchLine',
       'mode_pitchLength', 'balls_faced', 'strike_rate', 'boundary_rate',
       'dot_ball_pct', 'dismissal_ratio', 'avg_runs_death', 'avg_runs_middle',
       'avg_runs_powerplay', 'run_std', 'consistency_index',
       'avg_vs_left-arm_fast', 'avg_vs_left-arm_fast-medium',
       'avg_vs_left-arm_medium',
       'avg_vs_left-arm_medium,slow_left-arm_orthodox',
       'avg_vs_left-arm_medium-fast', 'avg_vs_left-arm_wrist-spin',
       'avg_vs_legbreak', 'avg_vs_legbreak_googly', 'avg_vs_right-arm_fast',
       'avg_vs_right-arm_fast-medium', 'avg_vs_right-arm_fast-medium,legbreak',
       'avg_vs_right-arm_fast-medium,legbreak_googly',
       'avg_vs_right-arm_medium', 'avg_vs_right-arm_medium,legbreak',
       'avg_vs_right-arm_medium,right-arm_offbreak',
       'avg_vs_right-arm_medium-fast',
       'avg_vs_right-arm_medium-fast,right-arm_offbr

In [157]:
def compute_pressure(row):
    if row['oversActual'] <= 6:
        phase_mult = 1.0   
    elif row['oversActual'] <= 15:
        phase_mult = 1.2   
    else:
        phase_mult = 1.5  


    pressure = ((row['totalWickets'] + 1) / (row['oversActual'] + 1)) * phase_mult
    return pressure

df['pressure_index'] = df.apply(compute_pressure, axis=1)


In [158]:
df.columns

Index(['Unnamed: 0', 'match_obj_id', 'inningNumber', 'oversActual',
       'pitchLine', 'pitchLength', 'isFour', 'isSix', 'isWicket', 'byes',
       'legbyes', 'wides', 'noballs', 'run', 'totalRuns', 'totalWickets',
       'shotType', 'time_of_day', 'Ground Name', 'Batsman_Name',
       'Batsman_Role', 'Full Name', 'Batsman_Batting_Style',
       'Batsman_Playing_Role', 'Bowler_Name', 'Bowler_Role',
       'Full Name_bowler', 'Batting Style (s)_bowler', 'Bowler_Bowling_Style',
       'Bowler_Playing_Role', 'Batsman_Name_clean', 'pitchLine_code',
       'pitchLength_code', 'isBoundary', 'phase', 'ball_type',
       'pressure_index'],
      dtype='object')

In [159]:
df = df.merge(
    batsman_stats[['avg_runs']],
    left_on='Batsman_Name_clean',
    right_index=True,
    how='left'
)

df['pressure_index_batsman'] = df['pressure_index'] * (1 - (df['run'] / (df['avg_runs'] + 1e-6)))


In [160]:
df.columns

Index(['Unnamed: 0', 'match_obj_id', 'inningNumber', 'oversActual',
       'pitchLine', 'pitchLength', 'isFour', 'isSix', 'isWicket', 'byes',
       'legbyes', 'wides', 'noballs', 'run', 'totalRuns', 'totalWickets',
       'shotType', 'time_of_day', 'Ground Name', 'Batsman_Name',
       'Batsman_Role', 'Full Name', 'Batsman_Batting_Style',
       'Batsman_Playing_Role', 'Bowler_Name', 'Bowler_Role',
       'Full Name_bowler', 'Batting Style (s)_bowler', 'Bowler_Bowling_Style',
       'Bowler_Playing_Role', 'Batsman_Name_clean', 'pitchLine_code',
       'pitchLength_code', 'isBoundary', 'phase', 'ball_type',
       'pressure_index', 'avg_runs', 'pressure_index_batsman'],
      dtype='object')

In [161]:
df_features.columns

Index(['Full Name', 'Run_Scored', 'Balls_Faced', 'Strike_Rate', 'Dismissals',
       'Dismissal_Rate', 'Boundary_percentage', 'Average', 'Full_Name_clean',
       'Batsman_Name_clean_x', 'total_runs', 'avg_runs', 'total_4s',
       'total_6s', 'total_wickets', 'total_boundaries', 'mode_pitchLine',
       'mode_pitchLength', 'balls_faced', 'strike_rate', 'boundary_rate',
       'dot_ball_pct', 'dismissal_ratio', 'Batsman_Name_clean_y',
       'best_ball_type'],
      dtype='object')

In [162]:
batsman_stats.columns

Index(['Batsman_Name_clean', 'total_runs', 'avg_runs', 'total_4s', 'total_6s',
       'total_wickets', 'total_boundaries', 'mode_pitchLine',
       'mode_pitchLength', 'balls_faced', 'strike_rate', 'boundary_rate',
       'dot_ball_pct', 'dismissal_ratio', 'avg_runs_death', 'avg_runs_middle',
       'avg_runs_powerplay', 'run_std', 'consistency_index',
       'avg_vs_left-arm_fast', 'avg_vs_left-arm_fast-medium',
       'avg_vs_left-arm_medium',
       'avg_vs_left-arm_medium,slow_left-arm_orthodox',
       'avg_vs_left-arm_medium-fast', 'avg_vs_left-arm_wrist-spin',
       'avg_vs_legbreak', 'avg_vs_legbreak_googly', 'avg_vs_right-arm_fast',
       'avg_vs_right-arm_fast-medium', 'avg_vs_right-arm_fast-medium,legbreak',
       'avg_vs_right-arm_fast-medium,legbreak_googly',
       'avg_vs_right-arm_medium', 'avg_vs_right-arm_medium,legbreak',
       'avg_vs_right-arm_medium,right-arm_offbreak',
       'avg_vs_right-arm_medium-fast',
       'avg_vs_right-arm_medium-fast,right-arm_offbr

In [163]:
df['balls_bowled'] = (df['oversActual'].astype(int) * 6) + ((df['oversActual'] % 1) * 10)
df['remaining_balls'] = 120 - df['balls_bowled']
df['remaining_overs'] = df['remaining_balls'] / 6
df['remaining_overs'] = df['remaining_overs'].clip(lower=0)


In [164]:
batsman_context = (
    df.groupby('Batsman_Name_clean')[['oversActual', 'remaining_overs', 'pressure_index_batsman']]
    .mean()
    .reset_index()
)
batsman_stats = batsman_stats.merge(batsman_context, on='Batsman_Name_clean', how='left')
print(batsman_stats.columns.tolist())


['Batsman_Name_clean', 'total_runs', 'avg_runs', 'total_4s', 'total_6s', 'total_wickets', 'total_boundaries', 'mode_pitchLine', 'mode_pitchLength', 'balls_faced', 'strike_rate', 'boundary_rate', 'dot_ball_pct', 'dismissal_ratio', 'avg_runs_death', 'avg_runs_middle', 'avg_runs_powerplay', 'run_std', 'consistency_index', 'avg_vs_left-arm_fast', 'avg_vs_left-arm_fast-medium', 'avg_vs_left-arm_medium', 'avg_vs_left-arm_medium,slow_left-arm_orthodox', 'avg_vs_left-arm_medium-fast', 'avg_vs_left-arm_wrist-spin', 'avg_vs_legbreak', 'avg_vs_legbreak_googly', 'avg_vs_right-arm_fast', 'avg_vs_right-arm_fast-medium', 'avg_vs_right-arm_fast-medium,legbreak', 'avg_vs_right-arm_fast-medium,legbreak_googly', 'avg_vs_right-arm_medium', 'avg_vs_right-arm_medium,legbreak', 'avg_vs_right-arm_medium,right-arm_offbreak', 'avg_vs_right-arm_medium-fast', 'avg_vs_right-arm_medium-fast,right-arm_offbreak', 'avg_vs_right-arm_offbreak', 'avg_vs_right-arm_offbreak,legbreak', 'avg_vs_right-arm_offbreak,legbreak_go

In [165]:
wicket_rate_zone = (
    df.groupby(['Batsman_Name_clean', 'pitchLine', 'pitchLength'])['isWicket']
      .mean()
      .reset_index()
      .rename(columns={'isWicket': 'wicket_rate_zone'})
)
df = df.merge(wicket_rate_zone, on=['Batsman_Name_clean', 'pitchLine', 'pitchLength'], how='left')
batsman_wicket_zone_avg = (
    wicket_rate_zone.groupby('Batsman_Name_clean')['wicket_rate_zone']
      .mean()
      .reset_index()
)
batsman_stats = batsman_stats.merge(batsman_wicket_zone_avg, on='Batsman_Name_clean', how='left')
print("✅ wicket_rate_zone feature added successfully!")
print(df[['Batsman_Name_clean', 'pitchLine', 'pitchLength', 'wicket_rate_zone']].head())


✅ wicket_rate_zone feature added successfully!
  Batsman_Name_clean      pitchLine             pitchLength  wicket_rate_zone
0           tony ura  ON_THE_STUMPS             GOOD_LENGTH          0.166667
1           tony ura  ON_THE_STUMPS  SHORT_OF_A_GOOD_LENGTH          0.000000
2           tony ura  ON_THE_STUMPS             GOOD_LENGTH          0.166667
3           tony ura  ON_THE_STUMPS             GOOD_LENGTH          0.166667
4           tony ura  ON_THE_STUMPS             GOOD_LENGTH          0.166667


In [166]:
df['phase_code'] = df['phase'].map({'Powerplay':0, 'Middle':1, 'Death':2})


In [167]:
X_features = [

    'avg_runs', 'strike_rate', 'boundary_rate', 'dot_ball_pct',
    'dismissal_ratio', 'run_std', 'consistency_index',

    
    'avg_runs_powerplay', 'avg_runs_middle', 'avg_runs_death',

    'balls_faced', 'oversActual', 'remaining_overs', 'pressure_index_batsman',

    
    'avg_runs_day', 'avg_runs_daynight', 'avg_runs_night',


    'avg_vs_left-arm_fast', 'avg_vs_left-arm_fast-medium', 'avg_vs_left-arm_medium',
    'avg_vs_right-arm_fast', 'avg_vs_right-arm_fast-medium',
    'avg_vs_right-arm_medium', 'avg_vs_right-arm_offbreak',
    'avg_vs_slow_left-arm_orthodox',

    
    'wicket_rate_zone'
]


In [168]:
batsman_stats.columns

Index(['Batsman_Name_clean', 'total_runs', 'avg_runs', 'total_4s', 'total_6s',
       'total_wickets', 'total_boundaries', 'mode_pitchLine',
       'mode_pitchLength', 'balls_faced', 'strike_rate', 'boundary_rate',
       'dot_ball_pct', 'dismissal_ratio', 'avg_runs_death', 'avg_runs_middle',
       'avg_runs_powerplay', 'run_std', 'consistency_index',
       'avg_vs_left-arm_fast', 'avg_vs_left-arm_fast-medium',
       'avg_vs_left-arm_medium',
       'avg_vs_left-arm_medium,slow_left-arm_orthodox',
       'avg_vs_left-arm_medium-fast', 'avg_vs_left-arm_wrist-spin',
       'avg_vs_legbreak', 'avg_vs_legbreak_googly', 'avg_vs_right-arm_fast',
       'avg_vs_right-arm_fast-medium', 'avg_vs_right-arm_fast-medium,legbreak',
       'avg_vs_right-arm_fast-medium,legbreak_googly',
       'avg_vs_right-arm_medium', 'avg_vs_right-arm_medium,legbreak',
       'avg_vs_right-arm_medium,right-arm_offbreak',
       'avg_vs_right-arm_medium-fast',
       'avg_vs_right-arm_medium-fast,right-arm_offbr

In [169]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

In [170]:
feature_cols = [
    'Batsman_Name_encoded', 'strike_rate', 'boundary_rate', 'dot_ball_pct',
    'dismissal_ratio', 'consistency_index', 'pressure_index_batsman',
    'avg_runs_powerplay', 'avg_runs_middle', 'avg_runs_death',
    'remaining_overs', 'avg_runs_day', 'avg_runs_daynight', 'avg_runs_night'
]

In [171]:
feature_cols += ['inningNumber', 'oversActual']

In [172]:
bowling_cols = [col for col in batsman_stats.columns if 'avg_vs_' in col]
feature_cols.extend(bowling_cols)

In [173]:
target_cols = ['mode_pitchLine', 'mode_pitchLength']

In [174]:
batsman_stats['Batsman_Name_clean'] = batsman_stats['Batsman_Name_clean'].astype(str).str.strip().str.lower()
df['Batsman_Name_clean'] = df['Batsman_Name_clean'].astype(str).str.strip().str.lower()

In [175]:
bowler_info = df[['Batsman_Name_clean', 'Bowler_Bowling_Style']].drop_duplicates()
merged = batsman_stats.merge(bowler_info, on='Batsman_Name_clean', how='inner')

In [176]:
from sklearn.preprocessing import LabelEncoder

In [177]:
le_bowler = LabelEncoder()
df['Bowler_Bowling_Style_encoded'] = le_bowler.fit_transform(df['Bowler_Bowling_Style'].astype(str))

In [178]:
bowling_type_models = {}

In [179]:
bowling_types = df['Bowler_Bowling_Style'].dropna().unique()

In [180]:

if 'Batsman_Name_encoded' not in batsman_stats.columns:
    le_batsman = LabelEncoder()
    batsman_stats['Batsman_Name_encoded'] = le_batsman.fit_transform(batsman_stats['Batsman_Name_clean'])


In [181]:
inning_info = (
    df.groupby('Batsman_Name_clean')['inningNumber']
    .agg(lambda x: x.mode()[0] if not x.mode().empty else np.nan)
    .rename('inningNumber')
    .reset_index()
)

batsman_stats = batsman_stats.merge(inning_info, on='Batsman_Name_clean', how='left')


In [182]:
merged = batsman_stats.merge(
    df[['Batsman_Name_clean', 'Bowler_Bowling_Style', 'Bowler_Bowling_Style_encoded']],
    on='Batsman_Name_clean',
    how='left'
)


In [183]:
feature_cols = [
    'Batsman_Name_encoded', 'strike_rate', 'boundary_rate', 'dot_ball_pct',
    'dismissal_ratio', 'consistency_index', 'pressure_index_batsman',
    'avg_runs_powerplay', 'avg_runs_middle', 'avg_runs_death',
    'remaining_overs', 'avg_runs_day', 'avg_runs_daynight', 'avg_runs_night',
    'inningNumber', 'oversActual', 'Bowler_Bowling_Style_encoded'
]
bowling_cols = [col for col in batsman_stats.columns if 'avg_vs_' in col]
feature_cols.extend(bowling_cols)

bowling_types = merged['Bowler_Bowling_Style'].dropna().unique()

In [187]:
for btype in bowling_types:
    subset = merged[merged['Bowler_Bowling_Style'] == btype]

    if len(subset) < 30:
        print(f" Skipping {btype} — insufficient data ({len(subset)})")
        continue

    # Prepare features and targets
    X = subset[feature_cols].fillna(0)
    y_line = subset['mode_pitchLine']
    y_length = subset['mode_pitchLength']
    X_train, X_test, y_train, y_test = train_test_split(X, y_line, test_size=0.2, random_state=42)
    model_line = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
    model_line.fit(X_train, y_train)
    acc_line = accuracy_score(y_test, model_line.predict(X_test))

    X_train, X_test, y_train, y_test = train_test_split(X, y_length, test_size=0.2, random_state=42)
    model_length = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
    model_length.fit(X_train, y_train)
    acc_len = accuracy_score(y_test, model_length.predict(X_test))

    print(f" {btype} — Line Model: {acc_line:.3f} | Length Model: {acc_len:.3f}")

    bowling_type_models[btype] = {
        'line_model': model_line,
        'length_model': model_length,
        'line_acc': acc_line,
        'length_acc': acc_len
    }

print(f"\n Trained {len(bowling_type_models)} bowling-type models successfully.")

 right-arm offbreak — Line Model: 1.000 | Length Model: 1.000
 right-arm fast-medium — Line Model: 0.996 | Length Model: 0.997
 slow left-arm orthodox — Line Model: 0.994 | Length Model: 0.998
 right-arm fast — Line Model: 0.996 | Length Model: 0.997
 right-arm medium-fast — Line Model: 0.994 | Length Model: 0.995
 legbreak — Line Model: 1.000 | Length Model: 0.998
 right-arm offbreak,legbreak — Line Model: 0.974 | Length Model: 0.987
 left-arm fast — Line Model: 0.992 | Length Model: 0.992
 left-arm fast-medium — Line Model: 1.000 | Length Model: 0.994
 left-arm medium-fast — Line Model: 1.000 | Length Model: 1.000
 right-arm medium — Line Model: 0.997 | Length Model: 1.000
 legbreak googly — Line Model: 0.985 | Length Model: 0.989
 left-arm medium — Line Model: 1.000 | Length Model: 0.943
 right-arm fast-medium,legbreak googly — Line Model: 0.933 | Length Model: 1.000
 left-arm wrist-spin — Line Model: 1.000 | Length Model: 1.000
 Skipping right-arm fast-medium,legbreak — insufficien

In [189]:
def recommend_optimal_strategy(batsman_name, phase):
    # Clean up any accidental spaces in column names
    batsman_stats.columns = batsman_stats.columns.str.strip()

    # --- Check if batsman exists ---
    if batsman_name not in batsman_stats['Batsman_Name_clean'].values:
        return {"Error": f"Batsman '{batsman_name}' not found in batsman_stats."}

    # Extract the batsman's row
    batsman_row = batsman_stats[batsman_stats['Batsman_Name_clean'] == batsman_name].iloc[0]

    # --- Validate phase ---
    phase_map = {
        'powerplay': 'avg_runs_powerplay',
        'middle': 'avg_runs_middle',
        'death': 'avg_runs_death'
    }
    if phase not in phase_map:
        return {"Error": f"Invalid phase '{phase}'. Choose from 'powerplay', 'middle', 'death'."}

    # --- Prepare input sample ---
    sample_df = pd.DataFrame([batsman_row[feature_cols]]).fillna(0)

    results = []

    # --- Predict for each bowling type ---
    for btype, models in bowling_type_models.items():
        try:
            line_pred = models['line_model'].predict(sample_df)[0]
            length_pred = models['length_model'].predict(sample_df)[0]

            #  Decode predictions using your mappings
            decoded_line = pitchLine_mapping.get(int(line_pred), f"Unknown({line_pred})")
            decoded_length = pitchLength_mapping.get(int(length_pred), f"Unknown({length_pred})")

            results.append({
                'Bowling Type': btype,
                'Predicted Line': decoded_line,
                'Predicted Length': decoded_length,
                'Line Model Accuracy': round(models['line_acc'], 3),
                'Length Model Accuracy': round(models['length_acc'], 3)
            })

        except Exception as e:
            print(f" Skipping {btype} due to error: {e}")
            continue

    # --- Return results ---
    if not results:
        return {"Error": "No valid predictions available for this batsman."}

    return {
        'Batsman': batsman_name,
        'Phase': phase,
        'Recommendations': results
    }

In [190]:
recommend_optimal_strategy('virat kohli','middle')

KeyError: "['Bowler_Bowling_Style_encoded'] not in index"